# Embedding spaces & dimensionality reduction (PCA, UMAP, t-SNE)

*0.2 Math / ML basics · run **Setup** first*

## Setup

Settings, a configured client, an `embed()` helper and numpy. Every cell below uses them.

In [1]:
"""Shared setup: typed settings, a configured client, numpy, and a print helper."""

import json

import numpy as np
from dotenv import find_dotenv
from openai import OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

np.set_printoptions(precision=4, suppress=True)


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    embedding_model: str = "text-embedding-3-small"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()
client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def embed(texts: list[str]) -> np.ndarray:
    """Embed texts with the configured model; returns one row per text."""
    response = client.embeddings.create(model=settings.embedding_model, input=texts)
    rows = []
    for item in response.data:
        rows.append(item.embedding)
    return np.array(rows)


def show(title: str, value) -> None:
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


print("chat model:", settings.openai_model, "| embedding model:", settings.embedding_model)

chat model: gpt-4o-mini | embedding model: text-embedding-3-small


### Embedding spaces

> **Problem.** Before building search on embeddings you want proof that the space is organised the way you assume — that "lion" sits near "cat", not near "truck" — and a number for how well it is organised.

**Idea.** In a good embedding space, nearest neighbours are semantically related; measure it with neighbour-label purity.

**Use when** choosing an embedding model, checking a fine-tuned one, debugging bad retrieval.  
**Not when** —.

```
              1,536-d space (cannot draw)
   cat ● dog ● lion ●        car ● truck ● bus ●        apple ● mango ● peach ●
        animals                    vehicles                    fruits
   3-NN purity: for each point, do its 3 nearest neighbours share its label?  →  1.00
```

**How it works.**
1. 24 words from three groups are embedded into 1,536 dimensions.
2. For three query words the three nearest neighbours by cosine are printed — all from the same group.
3. `neighbour_purity` fits a 3-nearest-neighbour classifier and asks what share of points are labelled correctly by their neighbours.
4. A purity near 1.0 means the space separates the groups cleanly; it is the number to compare across embedding models.

| | what happens | result |
|:--|:--|:--|
| ✓ good space | cat → kitten, dog, lion | purity ≈ 1.0 |
| ✗ poor space | cat → truck, mango | purity near chance (0.33) |

**Production code and its real output**

In [2]:
from sklearn.neighbors import KNeighborsClassifier

GROUPS = {
    "animal": ["cat", "dog", "horse", "lion", "rabbit", "eagle", "shark", "wolf"],
    "vehicle": ["car", "truck", "bicycle", "train", "airplane", "boat", "bus", "motorcycle"],
    "fruit": ["apple", "banana", "mango", "grape", "orange", "cherry", "peach", "lemon"],
}
words = []
labels = []
for group, members in GROUPS.items():
    for member in members:
        words.append(member)
        labels.append(group)
embeddings = embed(words)


def neighbour_purity(points: np.ndarray) -> float:
    """Share of points whose 3 nearest neighbours carry the same label."""
    model = KNeighborsClassifier(n_neighbors=4).fit(points, labels)
    return float(np.mean(model.predict(points) == np.array(labels)))


# Embedding spaces — nearby vectors are related meanings. Measured with nearest neighbours.
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)
for query in ["cat", "truck", "mango"]:
    index = words.index(query)
    order = np.argsort(-similarity[index])[1:4]
    neighbours = []
    for other in order:
        neighbours.append(f"{words[other]} ({similarity[index, other]:.2f})")
    print(f"{query:<7} nearest: " + ", ".join(neighbours))
print(
    "embedding shape:",
    embeddings.shape,
    "| 3-NN label purity in 1536-d:",
    neighbour_purity(embeddings),
)
assert neighbour_purity(embeddings) >= 0.9

cat     nearest: dog (0.60), rabbit (0.52), horse (0.52)
truck   nearest: train (0.60), boat (0.57), horse (0.54)
mango   nearest: cherry (0.48), peach (0.47), grape (0.46)
embedding shape: (24, 1536) | 3-NN label purity in 1536-d: 1.0


**What the output shows.** Every query's nearest neighbours came from its own group and 3-NN purity was 1.0 in the full 1,536-d space.

**In practice**
- **evaluate on your data** — purity on generic words says little about legal contracts or product SKUs; build a labelled set from your domain.
- **anisotropy** — many models pack all vectors into a narrow cone; raw cosines look uniformly high — rank, do not threshold.
- **multilingual** — check that translations land near each other if you serve more than one language.
- **fine-tuning moves the space** — after fine-tuning (layer 1.3), re-run this check — and re-embed everything in the index.

**Alternatives** — MTEB benchmark scores for model selection · retrieval recall@k on your own queries (layer 5.7)

**Terms** — *embedding space*: the set of all possible vectors from one model · *nearest neighbour*: the closest vector by the chosen metric · *purity*: share of neighbours with the same label


### PCA

> **Problem.** 1,536 dimensions cannot be plotted, and storing them costs 6 KB per text. You need a smaller number of dimensions that keeps as much of the structure as possible — and a way to know how much you lost.

**Idea.** Rotate the space so the first axes carry the most variance, then keep only those axes.

**Use when** quick visualisation, compressing vectors, de-noising before clustering.  
**Not when** the structure is curved — PCA is linear and will flatten clusters together (use UMAP).

```mermaid
flowchart LR
    E["1,536-d embeddings"] --> P["PCA: find directions · of greatest variance"] --> K["keep 2 axes"] --> V["2-d points · + explained variance"]
```

**How it works.**
1. `PCA(n_components=2).fit_transform(embeddings)` finds the two directions of greatest spread and projects every vector onto them.
2. `explained_variance_ratio_` says what share of the total variance those two axes keep.
3. The three group centres are printed in 2-d; purity after projection shows how much neighbourhood structure survived.
4. Fitting PCA with all components shows how many axes are needed for 90% of the variance — the compression you can get almost for free.

| | what happens | result |
|:--|:--|:--|
| ✓ 2 components | for a plot | groups still separable, some purity lost |
| ✓ ~90% variance | for storage | far fewer dimensions, little loss |
| ✗ curved manifolds | PCA is linear | clusters overlap |

**Production code and its real output**

In [3]:
from sklearn.neighbors import KNeighborsClassifier

GROUPS = {
    "animal": ["cat", "dog", "horse", "lion", "rabbit", "eagle", "shark", "wolf"],
    "vehicle": ["car", "truck", "bicycle", "train", "airplane", "boat", "bus", "motorcycle"],
    "fruit": ["apple", "banana", "mango", "grape", "orange", "cherry", "peach", "lemon"],
}
words = []
labels = []
for group, members in GROUPS.items():
    for member in members:
        words.append(member)
        labels.append(group)
embeddings = embed(words)


def neighbour_purity(points: np.ndarray) -> float:
    """Share of points whose 3 nearest neighbours carry the same label."""
    model = KNeighborsClassifier(n_neighbors=4).fit(points, labels)
    return float(np.mean(model.predict(points) == np.array(labels)))


# PCA — linear projection onto the directions of greatest variance. sklearn.decomposition.PCA.
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=0)
points = pca.fit_transform(embeddings)
print(
    "explained variance:",
    pca.explained_variance_ratio_,
    "total",
    round(float(pca.explained_variance_ratio_.sum()), 3),
)
for group in GROUPS:
    rows = np.array(labels) == group
    print(f"{group:<8} centre ({points[rows, 0].mean():+.2f}, {points[rows, 1].mean():+.2f})")
print("3-NN purity after PCA to 2-d:", neighbour_purity(points))

full = PCA(random_state=0).fit(embeddings)
print(
    "components for 90% variance:",
    int(np.searchsorted(np.cumsum(full.explained_variance_ratio_), 0.9)) + 1,
    "of",
    len(words) - 1,
)
assert neighbour_purity(points) >= 0.8

explained variance: [0.1496 0.1088] total 0.258
animal   centre (-0.13, -0.32)
vehicle  centre (-0.27, +0.26)
fruit    centre (+0.40, +0.06)
3-NN purity after PCA to 2-d: 1.0
components for 90% variance: 18 of 23


**What the output shows.** Two axes captured a modest share of the variance yet kept purity high; 90% of the variance needed far fewer than 1,536 dimensions.

**In practice**
- **deterministic and fast** — PCA has no random seed to tune and runs in milliseconds; use it first.
- **fit on a sample** — fit on 100k vectors, transform the rest; the components generalise.
- **Matryoshka embeddings** — models trained so the first N dimensions are already the best N — `dimensions=256` at request time beats PCA on those models.
- **whitening** — PCA can also decorrelate dimensions before clustering, which improves k-means.

**Alternatives** — UMAP / t-SNE for visualisation of clusters · product quantisation for storage compression (layer 4.6)

**Terms** — *principal component*: an axis of greatest variance · *explained variance*: share of total spread kept · *projection*: dropping onto fewer axes


### UMAP

> **Problem.** PCA's 2-d plot smears the clusters together because the real structure is curved. For a picture that shows the groups as groups — and for clustering — you need a method that preserves neighbourhoods rather than variance.

**Idea.** Build a graph of each point's nearest neighbours and lay it out in 2-d so neighbours stay neighbours.

**Use when** visualising clusters, pre-processing before density clustering (HDBSCAN), exploring a corpus.  
**Not when** you need distances between clusters to mean something, or a deterministic, fast transform for new points.

```
high-d neighbour graph            2-d layout
  cat—dog—lion …                  ●●●  animals
  car—truck—bus …        ──▶      ●●●  vehicles       neighbours kept, global distances not
  apple—mango …                   ●●●  fruits
```

**How it works.**
1. `UMAP(n_components=2, n_neighbors=5, min_dist=0.1)` connects each point to its 5 nearest neighbours in 1,536-d.
2. It then optimises 2-d positions so connected points stay close and unconnected ones are pushed apart.
3. Group centres and within-group spread are printed; the groups come out compact and well separated.
4. Purity after UMAP stays high — neighbourhoods survived the reduction.

| | what happens | result |
|:--|:--|:--|
| ✗ PCA | linear | clusters can overlap |
| ✓ UMAP | neighbour graph | clusters compact and separated |
| ✗ reading distances | between clusters | not meaningful |

**Production code and its real output**

In [4]:
from sklearn.neighbors import KNeighborsClassifier

GROUPS = {
    "animal": ["cat", "dog", "horse", "lion", "rabbit", "eagle", "shark", "wolf"],
    "vehicle": ["car", "truck", "bicycle", "train", "airplane", "boat", "bus", "motorcycle"],
    "fruit": ["apple", "banana", "mango", "grape", "orange", "cherry", "peach", "lemon"],
}
words = []
labels = []
for group, members in GROUPS.items():
    for member in members:
        words.append(member)
        labels.append(group)
embeddings = embed(words)


def neighbour_purity(points: np.ndarray) -> float:
    """Share of points whose 3 nearest neighbours carry the same label."""
    model = KNeighborsClassifier(n_neighbors=4).fit(points, labels)
    return float(np.mean(model.predict(points) == np.array(labels)))


# UMAP — non-linear; preserves local neighbourhoods, so clusters stay clusters. umap-learn.
import warnings

warnings.filterwarnings("ignore")
from umap import UMAP  # noqa: E402

points = UMAP(n_components=2, n_neighbors=5, min_dist=0.1, random_state=0).fit_transform(embeddings)
for group in GROUPS:
    rows = np.array(labels) == group
    spread = np.std(points[rows], axis=0).mean()
    print(
        
            f"{group:<8} centre ({points[rows, 0].mean():+.2f}, {points[rows, 1].mean():+.2f})  "
            f"spread {spread:.2f}"
        
    )
print("3-NN purity after UMAP to 2-d:", neighbour_purity(points))
assert neighbour_purity(points) >= 0.9

animal   centre (-12.95, +10.86)  spread 0.46
vehicle  centre (-10.67, +10.10)  spread 0.39
fruit    centre (+5.75, +11.25)  spread 0.47
3-NN purity after UMAP to 2-d: 1.0


**What the output shows.** The three groups formed tight, separated clusters with high purity; the printed spreads are small relative to the gaps between centres.

**In practice**
- **n_neighbors is the knob** — small values show local detail, large values global shape; 15 is the usual default.
- **set the seed** — results vary by run; fix `random_state` for reproducible figures.
- **cost** — UMAP is O(n log n) and fine to ~1M points; sample above that.
- **transform new points** — `fit` then `.transform(new)` works but is approximate; refit periodically.

**Alternatives** — t-SNE (older, slower, similar pictures) · PaCMAP · PCA for a first look

**Terms** — *neighbour graph*: each point linked to its nearest points · *manifold*: the curved surface the data lies on · *min_dist*: how tightly points may pack


### t-SNE

> **Problem.** You inherit analysis notebooks full of t-SNE plots and need to read them correctly: which features of the picture are real, and which are artefacts of the method.

**Idea.** Match neighbour probabilities in high-d and 2-d, so local neighbourhoods are faithful — and nothing else is.

**Use when** visualising local structure in a static report.  
**Not when** cluster sizes or distances between clusters matter; new points must be projected; datasets above ~50k points.

```
perplexity 5    ●●●   ●●●   ●●●     tight local groups
perplexity 20   ●●●●●●●●●●●●●●●     smoother, groups merge
                ─ layout changes, neighbourhoods mostly survive ─
```

**How it works.**
1. `TSNE(n_components=2, perplexity=5, init="pca")` converts pairwise distances into neighbour probabilities and finds a 2-d layout with matching probabilities.
2. Perplexity sets how many neighbours each point considers; it must be below the number of points.
3. Purity after t-SNE is high; a second run at perplexity 20 gives a different layout with similar purity.
4. Distances between clusters and cluster sizes in the plot are not meaningful — only who is near whom.

| | what happens | result |
|:--|:--|:--|
| ✓ local neighbours | who sits next to whom | reliable |
| ✗ cluster distance | gap between groups | meaningless |
| ✗ cluster size | spread of a group | meaningless |

**Production code and its real output**

In [5]:
from sklearn.neighbors import KNeighborsClassifier

GROUPS = {
    "animal": ["cat", "dog", "horse", "lion", "rabbit", "eagle", "shark", "wolf"],
    "vehicle": ["car", "truck", "bicycle", "train", "airplane", "boat", "bus", "motorcycle"],
    "fruit": ["apple", "banana", "mango", "grape", "orange", "cherry", "peach", "lemon"],
}
words = []
labels = []
for group, members in GROUPS.items():
    for member in members:
        words.append(member)
        labels.append(group)
embeddings = embed(words)


def neighbour_purity(points: np.ndarray) -> float:
    """Share of points whose 3 nearest neighbours carry the same label."""
    model = KNeighborsClassifier(n_neighbors=4).fit(points, labels)
    return float(np.mean(model.predict(points) == np.array(labels)))


# t-SNE — non-linear; keeps neighbours close, but cluster distances are not meaningful. sklearn.manifold.TSNE.
from sklearn.manifold import TSNE

points = TSNE(n_components=2, perplexity=5, init="pca", random_state=0).fit_transform(embeddings)
for group in GROUPS:
    rows = np.array(labels) == group
    print(f"{group:<8} centre ({points[rows, 0].mean():+.1f}, {points[rows, 1].mean():+.1f})")
print("3-NN purity after t-SNE to 2-d:", neighbour_purity(points))
other = TSNE(n_components=2, perplexity=20, init="pca", random_state=0).fit_transform(embeddings)
print(
    "purity at perplexity=20:",
    neighbour_purity(other),
    "(layout changes, neighbourhoods mostly survive)",
)
assert neighbour_purity(points) >= 0.9

animal   centre (+23.0, -19.8)
vehicle  centre (+19.5, -81.2)
fruit    centre (-12.7, +70.8)
3-NN purity after t-SNE to 2-d: 1.0
purity at perplexity=20: 0.9166666666666666 (layout changes, neighbourhoods mostly survive)


**What the output shows.** Purity stayed high at both perplexities while the layouts differed — neighbourhoods are the signal, positions are not.

**In practice**
- **perplexity 5–50** — try more than one; a conclusion that holds only at one perplexity is an artefact.
- **init with PCA** — `init="pca"` makes runs more stable than random initialisation.
- **slow** — O(n²) memory in the naive form; use UMAP for anything large.
- **no transform** — t-SNE cannot place new points; it is for one-off pictures.

**Alternatives** — UMAP (faster, can transform new points) · PCA

**Terms** — *perplexity*: effective number of neighbours considered · *layout*: the 2-d positions found
